In [ ]:
import torch.nn.functional as F
from torch import nn
import torch

class GlobalLayerNorm(nn.Module):
  """
    Calculate Global Layer Normalization
    dim: (int or list or torch.Size) - input shape from an expected input of size
    eps: a value added to the denominator for numerical stability.
    elementwise_affine: a boolean value that when set to True,
      this module has learnable per-element affine parameters
      initialized to ones (for weights) and zeros (for biases).
  """

  def __init__(self, dim, shape, eps=1e-8, elementwise_affine=True):
    super(GlobalLayerNorm, self).__init__()
    self.dim = dim
    self.eps = eps
    self.elementwise_affine = elementwise_affine

    if self.elementwise_affine:
      if shape == 3:
        self.weight = nn.Parameter(torch.ones(self.dim, 1))
        self.bias = nn.Parameter(torch.zeros(self.dim, 1))
      if shape == 4:
        self.weight = nn.Parameter(torch.ones(self.dim, 1, 1))
        self.bias = nn.Parameter(torch.zeros(self.dim, 1, 1))
    else:
      self.register_parameter('weight', None)
      self.register_parameter('bias', None)

  def forward(self, x):
    # x = N x C x K x S or N x C x L
    # N x 1 x 1
    # cln: mean,var N x 1 x K x S
    # gln: mean,var N x 1 x 1
    if x.dim() == 4:
      mean = torch.mean(x, (1, 2, 3), keepdim=True)
      var = torch.mean((x - mean) ** 2, (1, 2, 3), keepdim=True)
      if self.elementwise_affine:
        x = self.weight * (x - mean) / \
          torch.sqrt(var + self.eps) + self.bias
      else:
        x = (x - mean) / torch.sqrt(var + self.eps)
    if x.dim() == 3:
      mean = torch.mean(x, (1, 2), keepdim=True)
      var = torch.mean((x - mean) ** 2, (1, 2), keepdim=True)
      if self.elementwise_affine:
        x = self.weight * (x - mean) / \
          torch.sqrt(var + self.eps) + self.bias
      else:
        x = (x - mean) / torch.sqrt(var + self.eps)
    return x

class CumulativeLayerNorm(nn.LayerNorm):
  """
    Calculate Cumulative Layer Normalization
    dim: you want to norm dim
    elementwise_affine: learnable per-element affine parameters
  """

  def __init__(self, dim, elementwise_affine=True):
    super(CumulativeLayerNorm, self).__init__(dim, elementwise_affine=elementwise_affine, eps=1e-8)

  def forward(self, x):
    # x: N x C x K x S or N x C x L
    # N x K x S x C
    if x.dim() == 4:
      x = x.permute(0, 2, 3, 1).contiguous()
      # N x K x S x C == only channel norm
      x = super().forward(x)
      # N x C x K x S
      x = x.permute(0, 3, 1, 2).contiguous()
    if x.dim() == 3:
      x = torch.transpose(x, 1, 2)
      # N x L x C == only channel norm
      x = super().forward(x)
      # N x C x L
      x = torch.transpose(x, 1, 2)
    return x

def select_norm(norm, dim, shape):
  if norm == 'gln':
    return GlobalLayerNorm(dim, shape, elementwise_affine=True)
  if norm == 'cln':
    return CumulativeLayerNorm(dim, elementwise_affine=True)
  if norm == 'ln':
    return nn.GroupNorm(1, dim, eps=1e-8)
  else:
    return nn.BatchNorm1d(dim)

class Encoder(nn.Module):
  """
    Conv-Tasnet Encoder part
    kernel_size: the length of filters
    out_channels: the number of filters
  """

  def __init__(self, kernel_size=2, out_channels=64):
    super(Encoder, self).__init__()
    self.conv1d = nn.Conv1d(
      in_channels=1,
      out_channels=out_channels,
      kernel_size=kernel_size,
      stride=kernel_size // 2,
      groups=1,
      bias=False
    )

  def forward(self, x):
    """
      Input:
        x: [B, T], B is batch size, T is times
      Returns:
        x: [B, C, T_out]
        T_out is the number of time steps
    """
    x = torch.unsqueeze(x, dim=1) # B x T -> B x 1 x T
    x = self.conv1d(x) # B x 1 x T -> B x C x T_out
    x = F.relu(x)
    return x

class Decoder(nn.ConvTranspose1d):
  """
    Decoder of the TasNet
    This module can be seen as the gradient of Conv1d with respect to its input.
    It is also known as a fractionally-strided convolution
    or a deconvolution (although it is not an actual deconvolution operation).
  """

  def __init__(self, *args, **kwargs):
    super(Decoder, self).__init__(*args, **kwargs)

  def forward(self, x):
    """
    x: [B, N, L]
    """
    if x.dim() not in [2, 3]:
      raise RuntimeError("{} accept 3/4D tensor as input".format(self.__name__))
    x = super().forward(x if x.dim() == 3 else torch.unsqueeze(x, 1))

    if torch.squeeze(x).dim() == 1:
      x = torch.squeeze(x, dim=1)
    else:
      x = torch.squeeze(x)
    return x

In [2]:
class Dual_RNN_Block(nn.Module):
  """
    Implementation of the intra-RNN and the inter-RNN
    input:
        in_channels: The number of expected features in the input x
        out_channels: The number of features in the hidden state h
        rnn_type: RNN, LSTM, GRU
        norm: gln = "Global Norm", cln = "Cumulative Norm", ln = "Layer Norm"
        dropout: If non-zero, introduces a Dropout layer on the outputs
                  of each LSTM layer except the last layer,
                  with dropout probability equal to dropout. Default: 0
        bidirectional: If True, becomes a bidirectional LSTM. Default: False
  """

  def __init__(self, out_channels, hidden_channels, rnn_type='LSTM', norm='ln', dropout=0, bidirectional=False, num_spks=2):
    super(Dual_RNN_Block, self).__init__()
    # RNN model
    self.intra_rnn = getattr(nn, rnn_type)(out_channels, hidden_channels, 1, batch_first=True, dropout=dropout, bidirectional=bidirectional)
    self.inter_rnn = getattr(nn, rnn_type)(out_channels, hidden_channels, 1, batch_first=True, dropout=dropout, bidirectional=bidirectional)
    # Norm
    self.intra_norm = select_norm(norm, out_channels, 4)
    self.inter_norm = select_norm(norm, out_channels, 4)
    # Linear
    self.intra_linear = nn.Linear(hidden_channels * 2 if bidirectional else hidden_channels, out_channels)
    self.inter_linear = nn.Linear(hidden_channels * 2 if bidirectional else hidden_channels, out_channels)

  def forward(self, x):
    """
      x: [B, N, K, S]
      out: [Spks, B, N, K, S]
    """
    B, N, K, S = x.shape
    # intra RNN
    intra_rnn = x.permute(0, 3, 2, 1).contiguous().view(B * S, K, N) # [BS, K, N]
    intra_rnn, _ = self.intra_rnn(intra_rnn) # [BS, K, H]
    intra_rnn = self.intra_linear(intra_rnn.contiguous().view(B * S * K, -1)).view(B * S, K, -1) # [BS, K, N]
    intra_rnn = intra_rnn.view(B, S, K, N) # [B, S, K, N]
    intra_rnn = intra_rnn.permute(0, 3, 2, 1).contiguous() # [B, N, K, S]
    intra_rnn = self.intra_norm(intra_rnn)

    # [B, N, K, S]
    intra_rnn = intra_rnn + x

    # inter RNN
    inter_rnn = intra_rnn.permute(0, 2, 3, 1).contiguous().view(B * K, S, N) # [BK, S, N]
    inter_rnn, _ = self.inter_rnn(inter_rnn) # [BK, S, H]
    inter_rnn = self.inter_linear(inter_rnn.contiguous().view(B * S * K, -1)).view(B * K, S, -1) # [BK, S, N]
    inter_rnn = inter_rnn.view(B, K, S, N) # [B, K, S, N]
    inter_rnn = inter_rnn.permute(0, 3, 1, 2).contiguous() # [B, N, K, S]
    inter_rnn = self.inter_norm(inter_rnn)
    
    out = inter_rnn + intra_rnn # [B, N, K, S]
    return out

class Dual_Path_RNN(nn.Module):
  """
    Implementation of the Dual-Path-RNN model
    input:
        in_channels: The number of expected features in the input x
        out_channels: The number of features in the hidden state h
        rnn_type: RNN, LSTM, GRU
        norm: gln = "Global Norm", cln = "Cumulative Norm", ln = "Layer Norm"
        dropout: If non-zero, introduces a Dropout layer on the outputs
                  of each LSTM layer except the last layer,
                  with dropout probability equal to dropout. Default: 0
        bidirectional: If True, becomes a bidirectional LSTM. Default: False
        num_layers: number of Dual-Path-Block
        K: the length of chunk
        num_spks: the number of speakers
  """

  def __init__(self, in_channels, out_channels, hidden_channels, rnn_type='LSTM', norm='ln', dropout=0, bidirectional=False, num_layers=4, K=200, num_spks=2):
    super(Dual_Path_RNN, self).__init__()
    self.K = K
    self.num_spks = num_spks
    self.num_layers = num_layers
    self.norm = select_norm(norm, in_channels, 3)
    self.conv1d = nn.Conv1d(in_channels, out_channels, 1, bias=False)

    self.dual_rnn = nn.ModuleList([])
    for i in range(num_layers):
      self.dual_rnn.append(Dual_RNN_Block(out_channels, hidden_channels,
                                          rnn_type=rnn_type, norm=norm, dropout=dropout,
                                          bidirectional=bidirectional))

    self.conv2d = nn.Conv2d(out_channels, out_channels * num_spks, kernel_size=1)
    self.end_conv1x1 = nn.Conv1d(out_channels, in_channels, 1, bias=False)
    self.prelu = nn.PReLU()
    self.activation = nn.ReLU()
    # gated output layer
    self.output = nn.Sequential(
      nn.Conv1d(out_channels, out_channels, 1),
      nn.Tanh()
    )
    self.output_gate = nn.Sequential(
       nn.Conv1d(out_channels, out_channels, 1),
       nn.Sigmoid()
    )

  def forward(self, x):
    """
      x: [B, N, L]
    """
    x = self.norm(x) # [B, N, L]    
    x = self.conv1d(x) # [B, N, L]
    x, gap = self._Segmentation(x, self.K) # [B, N, K, S]
    
    for i in range(self.num_layers): # [B, N*spks, K, S]
      x = self.dual_rnn[i](x)
    
    x = self.prelu(x)
    x = self.conv2d(x)
    
    B, _, K, S = x.shape
    x = x.view(B * self.num_spks, -1, K, S) # [B*spks, N, K, S]
    
    x = self._over_add(x, gap) # [B*spks, N, L]
    x = self.output(x) * self.output_gate(x)

    x = self.end_conv1x1(x)  # [spks*B, N, L]
    _, N, L = x.shape
    x = x.view(B, self.num_spks, N, L) # [B*spks, N, L] -> [B, spks, N, L]
    x = self.activation(x)
    x = x.transpose(0, 1) # [spks, B, N, L]

    return x

  def _padding(self, input, K):
    """
      padding the audio times
      K: chunks of length
      P: hop size
      input: [B, N, L]
    """
    B, N, L = input.shape
    P = K // 2
    gap = K - (P + L % K) % K
    if gap > 0:
      pad = torch.Tensor(torch.zeros(B, N, gap)).type(input.type())
      input = torch.cat([input, pad], dim=2)

    _pad = torch.Tensor(torch.zeros(B, N, P)).type(input.type())
    input = torch.cat([_pad, input, _pad], dim=2)

    return input, gap

  def _Segmentation(self, input, K):
    """
      the segmentation stage splits
      K: chunks of length
      P: hop size
      input: [B, N, L]
      output: [B, N, K, S]
    """
    B, N, L = input.shape
    P = K // 2
    input, gap = self._padding(input, K)
    # [B, N, K, S]
    input1 = input[:, :, :-P].contiguous().view(B, N, -1, K)
    input2 = input[:, :, P:].contiguous().view(B, N, -1, K)
    input = torch.cat([input1, input2], dim=3).view(B, N, -1, K).transpose(2, 3)

    return input.contiguous(), gap

  def _over_add(self, input, gap):
    """
      Merge sequence
      input: [B, N, K, S]
      gap: padding length
      output: [B, N, L]
    """
    B, N, K, S = input.shape
    P = K // 2
    input = input.transpose(2, 3).contiguous().view(B, N, -1, K * 2) # [B, N, S, K]

    input1 = input[:, :, :, :K].contiguous().view(B, N, -1)[:, :, P:]
    input2 = input[:, :, :, K:].contiguous().view(B, N, -1)[:, :, :-P]
    input = input1 + input2

    if gap > 0:
      input = input[:, :, :-gap] # [B, N, L]

    return input

class DPRNN(nn.Module):
  """
    model of Dual Path RNN
    input:
      in_channels: The number of expected features in the input x
      out_channels: The number of features in the hidden state h
      hidden_channels: The hidden size of RNN
      kernel_size: Encoder and Decoder Kernel size
      rnn_type: RNN, LSTM, GRU
      norm: gln = "Global Norm", cln = "Cumulative Norm", ln = "Layer Norm"
      dropout: If non-zero, introduces a Dropout layer on the outputs
                of each LSTM layer except the last layer,
                with dropout probability equal to dropout. Default: 0
      bidirectional: If True, becomes a bidirectional LSTM. Default: False
      num_layers: number of Dual-Path-Block
      K: the length of chunk
      num_spks: the number of speakers
  """

  def __init__(self, in_channels, out_channels, hidden_channels,
                kernel_size=2, rnn_type='LSTM', norm='ln', dropout=0,
                bidirectional=False, num_layers=4, K=200, num_spks=2):
    super(DPRNN, self).__init__()
    self.encoder = Encoder(kernel_size=kernel_size, out_channels=in_channels)
    self.separation = Dual_Path_RNN(
      in_channels,
      out_channels,
      hidden_channels,
      rnn_type=rnn_type, norm=norm, dropout=dropout,
      bidirectional=bidirectional, num_layers=num_layers, K=K, num_spks=num_spks
    )
    self.decoder = Decoder(
      in_channels=in_channels,
      out_channels=1,
      kernel_size=kernel_size,
      stride=kernel_size // 2,
      bias=False
    )
    self.num_spks = num_spks

  def forward(self, x):
    """
      x: [B, L]
    """
    e = self.encoder(x) # [B, N, L]
    s = self.separation(e)  # [spks, B, N, L]
    out = [s[i] * e for i in range(self.num_spks)] # [B, N, L] -> [B, L]
    audio = [self.decoder(out[i]) for i in range(self.num_spks)]
    audio = torch.stack(audio, dim=1)  # (B, n_spk, T)
    return audio

Prepare the dataloader

In [3]:
from torch.utils.data import Dataset
import torchaudio
import os

class SpeechSeparationDataset(Dataset):
  def __init__(self, root_dir, mix_folder='mix_clean', s1_folder='s1', s2_folder='s2', sample_rate=8000, num_samples=40000):
    self.mix_dir = os.path.join(root_dir, mix_folder)
    self.s1_dir = os.path.join(root_dir, s1_folder)
    self.s2_dir = os.path.join(root_dir, s2_folder)
    self.file_names = sorted(os.listdir(self.mix_dir))
    self.sample_rate = sample_rate
    self.num_samples = num_samples

  def __len__(self):
    return len(self.file_names)

  def __getitem__(self, idx):
    mix_path = os.path.join(self.mix_dir, self.file_names[idx])
    s1_path = os.path.join(self.s1_dir, self.file_names[idx])
    s2_path = os.path.join(self.s2_dir, self.file_names[idx])

    mix, _ = torchaudio.load(mix_path)
    s1, _ = torchaudio.load(s1_path)
    s2, _ = torchaudio.load(s2_path)

    # --- Crop or pad each signal to num_samples ---
    mix = self._fix_length(mix, self.num_samples)
    s1 = self._fix_length(s1, self.num_samples)
    s2 = self._fix_length(s2, self.num_samples)

    # Stack sources for target: shape (2, num_samples)
    target = torch.cat([s1, s2], dim=0)
    return mix, target  # mix: (1, num_samples), target: (2, num_samples)

  @staticmethod
  def _fix_length(wav, num_samples):
    """
      Crop or pad the waveform to num_samples (on last dimension).
      wav: tensor shape (1, T)
    """
    T = wav.shape[-1]
    if T > num_samples:
      wav = wav[..., :num_samples]
    elif T < num_samples:
      pad_shape = list(wav.shape)
      pad_shape[-1] = num_samples - T
      pad = torch.zeros(*pad_shape, dtype=wav.dtype)
      wav = torch.cat([wav, pad], dim=-1)
    return wav

In [13]:
from torch.utils.data import DataLoader, Subset

path = './Dataset/TestLibri'
num_samples = 8000  # num_samples / rate = duration (seconds)
num_dataset = 10

def collate_fn(batch):
  # Optionally, pad to the longest in batch
  # Here, we simply stack as is (assuming same length)
  mixs, targets = zip(*batch)
  mixs = torch.stack(mixs)  # (B, 1, T)
  targets = torch.stack(targets)  # (B, 2, T)
  return mixs.squeeze(1), targets  # (B, T), (B, 2, T)

dataset = SpeechSeparationDataset(
  path,
  mix_folder='mix',
  num_samples=num_samples
)

small_dataset = Subset(
  dataset,
  range(num_dataset)
)

dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

small_dataloader = DataLoader(small_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

Define the loss function

In [5]:
def si_snr_loss(estimate, target, eps=1e-8):
  """
  estimate: (B, n_spk, T)
  target: (B, n_spk, T)
  """
  def si_snr(s_hat, s):
    s = s - s.mean(dim=-1, keepdim=True)
    s_hat = s_hat - s_hat.mean(dim=-1, keepdim=True)
    pair_wise_dot = torch.sum(s_hat * s, dim=-1, keepdim=True)
    s_energy = torch.sum(s ** 2, dim=-1, keepdim=True) + eps
    proj = pair_wise_dot * s / s_energy
    noise = s_hat - proj
    ratio = torch.sum(proj ** 2, dim=-1) / (torch.sum(noise ** 2, dim=-1) + eps)
    return 10 * torch.log10(ratio + eps)

  # Permutation Invariant Training (PIT)
  si_snr_1 = si_snr(estimate[:,0], target[:,0]) + si_snr(estimate[:,1], target[:,1])
  si_snr_2 = si_snr(estimate[:,0], target[:,1]) + si_snr(estimate[:,1], target[:,0])
  # maximize si_snr, so minimize -si_snr
  loss = -torch.mean(torch.max(si_snr_1, si_snr_2))
  return loss

In [6]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
import torch.optim as optim

dprnn = DPRNN(
  in_channels=256,
  out_channels=64,
  hidden_channels=128,
  kernel_size=2,
  rnn_type='LSTM',
  norm='ln',
  dropout=0.0,
  bidirectional=True,
  num_layers=6,
  K=250,
  num_spks=2
).to(device)

optimizer = optim.Adam(dprnn.parameters(), lr=1e-3)

Test the model output

In [ ]:
B, T = 1, 8000  # batch, samples
x = torch.randn(B, T).to(device)
out = dprnn(x)
print(out.shape)  # [B, T, F]

torch.Size([1, 2, 8000])


Start the training loop

In [14]:
num_epochs = 2

for epoch in range(num_epochs):
  dprnn.train()
  running_loss = 0.0

  print(f"Epoch: {epoch+1}/{num_epochs}")
  i = 0
  for mix, target in dataloader:
    mix = mix.to(device)  # (B, T)
    target = target.to(device)  # (B, 2, T)
    optimizer.zero_grad()
    estimate = dprnn(mix)  # (B, 2, T)
    loss = si_snr_loss(estimate, target)
    print(f"{i + 1}: {loss}")
    loss.backward()
    optimizer.step()
    running_loss += loss.item()
    i += 1
    
  print(f"Epoch {epoch+1}, Loss: {running_loss/len(dataloader):.4f}")

Epoch: 1/2
1: 8.580747604370117
2: 3.0543527603149414
3: 0.49003779888153076
4: 1.256456971168518
5: 0.9322757720947266
6: -1.1710453033447266
7: 3.9811811447143555
8: 0.7570011615753174
9: -0.9925532341003418
10: 0.6732475757598877
Epoch 1, Loss: 1.7562
Epoch: 2/2
1: 0.5175317525863647
2: -1.297561526298523
3: 0.1503230333328247
4: 0.43983888626098633
5: 0.2779226303100586
6: 2.149543285369873
7: -0.5070737600326538
8: 0.231156587600708
9: -1.6628594398498535
10: 0.19240808486938477
Epoch 2, Loss: 0.0491


In [25]:
import IPython.display as ipd

mix_test, source_test = dataset[3]

dprnn.eval()
with torch.no_grad():
  out = dprnn(mix_test.to(device)).squeeze(0).cpu().numpy()

  ipd.display(ipd.Audio(data=mix_test.squeeze(0).numpy(), rate=8000))
  ipd.display(ipd.Audio(data=out[0], rate=8000))
  ipd.display(ipd.Audio(data=out[1], rate=8000))